# 107 — Knowledge graphs y GraphRAG

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.**
`(Alan Turing, PROPUSO, máquina de Turing) {año: 1936}`,
`(Alan Turing, TRABAJÓ_EN, Bletchley Park)`,
`(Alan Turing, DESCIFRÓ, Enigma)`,
`(Gordon Welchman, DESCIFRÓ, Enigma)`,
`(Gordon Welchman, TRABAJÓ_EN, Bletchley Park)` (implícita en "donde… junto a").
Correferencias resueltas: "Durante la guerra trabajó" → el sujeto omitido es Turing;
"donde" → Bletchley Park; "junto a" duplica la relación para Welchman.

**Ejercicio 2.** "¿Qué pares de científicos distintos trabajaron en el mismo lugar?"
Dos saltos: `a → lugar ← b` (el segundo invertido). En texto plano exigiría cruzar
frases que quizá están en documentos distintos.

**Ejercicio 3.** a) **Vectorial**: la respuesta vive en un pasaje del manual.
b) **Grafo**: intersección relacional (empleado —TRABAJÓ_EN→ proyecto {estado:
cancelado}); el top-k de chunks no garantiza cubrir los tres proyectos.
c) **GraphRAG global**: agregación temática sobre todo el corpus → comunidades +
resúmenes jerárquicos; ningún top-k local la responde.

**Ejercicio 4.** El contrato se verifica en el código: `kind == "logic"` y `evidence`
no vacía.

In [ ]:
result = run_lab("logic", seed=107)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


In [ ]:
tripletas = [
    ("Alan Turing", "PROPUSO", "máquina de Turing", {"año": 1936}),
    ("Alan Turing", "TRABAJÓ_EN", "Bletchley Park", {}),
    ("Alan Turing", "DESCIFRÓ", "Enigma", {}),
    ("Gordon Welchman", "DESCIFRÓ", "Enigma", {}),
    ("Gordon Welchman", "TRABAJÓ_EN", "Bletchley Park", {}),
]
for t in tripletas:
    print(t)

# Consulta multi-hop sobre las tripletas: ¿quién trabajó donde trabajó Turing?
lugares_turing = {o for s, p, o, _ in tripletas
                  if s == "Alan Turing" and p == "TRABAJÓ_EN"}
colegas = {s for s, p, o, _ in tripletas
           if p == "TRABAJÓ_EN" and o in lugares_turing and s != "Alan Turing"}
print("Colegas de lugar de Turing:", colegas)

eleccion = {
    "a": "vectorial — la respuesta vive en un pasaje del manual",
    "b": "grafo — intersección relacional sobre tres proyectos",
    "c": "GraphRAG global — agregación temática de todo el corpus",
}
for k, v in eleccion.items():
    print(k, "→", v)

## Reflexión

1. ¿Por qué la pregunta "¿cuáles son los tres temas dominantes de este corpus?" es inalcanzable para un RAG vectorial top-k, y qué componente de GraphRAG la hace posible?
2. Si la resolución de entidades falla y "Meta" y "Facebook Inc." quedan como nodos separados, ¿qué tipo de consultas devuelven resultados incompletos sin dar ningún error?
3. El grafo extraído por LLM tiene apariencia de base de datos pero hereda alucinaciones del modelo. ¿Qué proceso de verificación propondrías antes de tratar una tripleta como hecho?